In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../')
import torch
from neural_control.extended_controllers import NNCDIController, FCController
from neural_control.extended_dynamics import SequentialDualSourcing
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from plotly import graph_objects as go
from sourcing_models.utilities import sample_trajectories_capped_dual_index, \
sample_trajectories_dual_index, sample_trajectories_single_index, sample_trajectories_tailored_base_surge
import time
import learn2learn as l2l

from neural_control.experiments.helpers import load_experiment_conf

import matplotlib.pyplot as plt
from matplotlib import rcParams
from utilities import magic_random_seed

sourcing_parameters_path = '../../sourcing_models/recursion_input_files/ds1_lr=2_b=95_h=5_u08.in'
experiment_data = load_experiment_conf(sourcing_parameters_path)
total_time = 100
torch.set_default_tensor_type('torch.cuda.FloatTensor')


sourcing_parameters = dict(ce=int(experiment_data.c_e),
                           cr=int(experiment_data.c_r),
                           le=int(experiment_data.l_e),
                           lr=int(experiment_data.l_r),
                           h=int(experiment_data.h),
                           b=int(experiment_data.b),
                           T=total_time,
                           fe = 0,#int(experiment_data.f_e),
                           fr = 0,#int(experiment_data.f_r)

                          )

sourcing_parameters = dict(ce=5,
                           cr=0,
                           le=0,
                           lr=2,
                           h=5,
                           b=95,
                           T=100,
                           fe=100,
                           fr=50
                          
                          
                          )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
#Ioannis has some 85 files here wheereas paper has only 95?
sourcing_parameters

{'ce': 5,
 'cr': 0,
 'le': 0,
 'lr': 2,
 'h': 5,
 'b': 95,
 'T': 100,
 'fe': 100,
 'fr': 50}

In [4]:
class HypergradTransform(torch.nn.Module):
    """Hypergradient-style per-parameter learning rates"""

    def __init__(self, param, lr=0.00001):
        super(HypergradTransform, self).__init__()
        self.lr = lr * torch.ones_like(param, requires_grad=True)
        self.lr = torch.nn.Parameter(self.lr)

    def forward(self, grad):
        #print(torch.isnan(grad).any())
        return self.lr * grad


In [5]:
### We use celu non-linearities for the input layer, hidden layers, and output layer

#fcc = NNCDIController(lr=sourcing_parameters['lr'], 

### One can experiment with different layer architectures and activations per layer
nnc_hyperparameters = dict(
    n_hidden_units = [64, 32, 16, 8, 4]
)

### We use celu non-linearities for the input layer, hidden layers, and output layer
nnc_hyperparameters['n_activations'] = [torch.nn.CELU(alpha=1)]*(2 + len(nnc_hyperparameters['n_hidden_units']))
fcc = FCController(lr=sourcing_parameters['lr'], 
                   le=sourcing_parameters['le'], 
                   n_hidden_units=nnc_hyperparameters['n_hidden_units'],
                   activations=nnc_hyperparameters['n_activations']
) # controller neural network object
dsd = SequentialDualSourcing(fcc, I_0=6, **sourcing_parameters) # Dual Sourcing Dynamics object

best_loss = [np.infty]
best_model = [None]



metaopt = l2l.optim.LearnableOptimizer(
        model=dsd,  # We pass the model, not its parameters
        transform=HypergradTransform,  # Any transform could work
        lr=1e-6)
metaopt.to(torch.zeros(1).device)


# optimizer2 = torch.optim.RMSprop([dsd.I_0],
#                                 #list(fcc.parameters()), 
#                                 lr=1e-3
#                                )



LearnableOptimizer(
  (transforms): ModuleList(
    (0): HypergradTransform()
    (1): HypergradTransform()
    (2): HypergradTransform()
    (3): HypergradTransform()
    (4): HypergradTransform()
    (5): HypergradTransform()
    (6): HypergradTransform()
    (7): HypergradTransform()
    (8): HypergradTransform()
    (9): HypergradTransform()
    (10): HypergradTransform()
    (11): HypergradTransform()
    (12): HypergradTransform()
  )
)

In [8]:
optimizer = torch.optim.RMSprop(#[dsd.I_0],
                                #list(fcc.parameters()) + [dsd.I_0], 
                                metaopt.parameters(),
                                lr=1e-3
                               )
metaopt = l2l.optim.LearnableOptimizer(
        model=dsd,  # We pass the model, not its parameters
        transform=HypergradTransform,  # Any transform could work
        lr=1e-3)
metaopt.to(torch.zeros(1).device)

LearnableOptimizer(
  (transforms): ModuleList(
    (0): HypergradTransform()
    (1): HypergradTransform()
    (2): HypergradTransform()
    (3): HypergradTransform()
    (4): HypergradTransform()
    (5): HypergradTransform()
    (6): HypergradTransform()
    (7): HypergradTransform()
    (8): HypergradTransform()
    (9): HypergradTransform()
    (10): HypergradTransform()
    (11): HypergradTransform()
    (12): HypergradTransform()
  )
)

In [9]:
fine_tuning_iterations = 3000
minibatch_size = 1024
random_seed = 6

all_training_costs = []

t0 = time.time()
for it in range(fine_tuning_iterations):
    torch.manual_seed(magic_random_seed)
    dsd.train()
    
    metaopt.zero_grad()
    optimizer.zero_grad()

    dsd.reset(minibatch_size, seed=magic_random_seed)
    nn_context = {}
    total_costs = 0
    for i in range(dsd.T):
        current_costs, demands, current_inventories, qr, qra, qe, qea, nn_context = dsd.simulate()
        total_costs = current_costs.mean() + total_costs
    all_training_costs.append(float(total_costs)/dsd.T)
    total_costs.backward()
    if it % 20 == 0:
        print(total_costs/dsd.T)
    if total_costs < best_loss[0]:
        best_loss[0] = total_costs.detach().cpu().item()
        best_model[0] = deepcopy(fcc.state_dict())
    
    #if it > 1 and all_training_costs[-1] <= 26.4:
    #        break

    optimizer.step()
    metaopt.step()
t1 = time.time()
total = t1-t0
print(total)

tensor(11478.7568, grad_fn=<DivBackward0>)
tensor(666.0441, grad_fn=<DivBackward0>)
tensor(404.2985, grad_fn=<DivBackward0>)
tensor(324.0144, grad_fn=<DivBackward0>)
tensor(279.6328, grad_fn=<DivBackward0>)
tensor(246.5528, grad_fn=<DivBackward0>)
tensor(232.5228, grad_fn=<DivBackward0>)
tensor(219.8367, grad_fn=<DivBackward0>)
tensor(206.9486, grad_fn=<DivBackward0>)
tensor(199.0745, grad_fn=<DivBackward0>)
tensor(193.6733, grad_fn=<DivBackward0>)
tensor(185.9317, grad_fn=<DivBackward0>)
tensor(181.4318, grad_fn=<DivBackward0>)
tensor(178.0705, grad_fn=<DivBackward0>)
tensor(172.8735, grad_fn=<DivBackward0>)
tensor(168.6488, grad_fn=<DivBackward0>)
tensor(166.5969, grad_fn=<DivBackward0>)
tensor(165.0232, grad_fn=<DivBackward0>)
tensor(161.6670, grad_fn=<DivBackward0>)
tensor(158.8287, grad_fn=<DivBackward0>)
tensor(157.4098, grad_fn=<DivBackward0>)
tensor(153.6944, grad_fn=<DivBackward0>)
tensor(153.7585, grad_fn=<DivBackward0>)
tensor(152.2209, grad_fn=<DivBackward0>)
tensor(148.911

In [13]:
#list(metaopt.parameters())

In [ ]:
list(dsd.named_parameters())

In [ ]:
dsd.I_0

In [14]:
LearnableOptimizer?

Object `LearnableOptimizer` not found.


In [16]:
l2l.optim.LearnableOptimizer??

Init signature: l2l.optim.LearnableOptimizer(model, transform, lr=1.0)
Source:        
class LearnableOptimizer(torch.nn.Module):
    """
    [[Source]](https://github.com/learnables/learn2learn/blob/master/learn2learn/optim/learnable_optimizer.py)

    **Description**

    A PyTorch Optimizer with learnable transform, enabling the implementation
    of meta-descent / hyper-gradient algorithms.

    This optimizer takes a Module and a gradient transform.
    At each step, the gradient of the module is passed through the transforms,
    and the module differentiably update -- i.e. when the next backward is called,
    gradients of both the module and the transform are computed.
    In turn, the transform can be updated via your favorite optmizer.

    **Arguments**

    * **model** (Module) - Module to be updated.
    * **transform** (Module) - Transform used to compute updates of the model.
    * **lr** (float) - Learning rate.

    **References**

    1. Sutton. 1992. “Gain Adaptation